# 📜 Citadel Access Contracts - Testing Center

## Overview

Provision and validate **multiple Access Contracts** with different configurations and targets, each
backed by a **dynamically generated APIM product policy** that enforces:

- **Model-level RBAC** via the `allowedModels` variable (per-contract list of allowed model names)
- **Capacity allocation** via `llm-token-limit` (`tokens-per-minute`, `token-quota`, `token-quota-period`)
- **Optional Azure Key Vault integration** (resolve endpoint + API key secrets)
- **Optional Microsoft Foundry connection integration** (auto-create a Foundry project connection using the
  new default **`ProjectManagedIdentity`** auth: managed-identity **JWT** + `api-key` **custom header**)

| Access Contract | Configuration | Description | Target Runtime |
|---|---|---|---|
| Sales-Assistant | Key Vault integration only | Sales Assistant workload with secrets resolved from Key Vault. | Agents running on Azure (AKS, ACA, App Service) |
| HR-ChatAgent | Key Vault + Foundry connection integrations | HR chat agent flow with a `ProjectManagedIdentity` Foundry connection (JWT + api-key header) and matching JWT validation policy. | Foundry Agents |
| Support-Bot | Direct output (no Key Vault nor Foundry integration) | Support bot scenario using direct output without external integrations. | Custom hosted agents |

## Expected outcomes

- Three Access Contracts (APIM products + subscriptions) deployed via Bicep
- A per-contract APIM product policy (`ai-product-policy.xml`) generated **dynamically** from
  the notebook variables (model RBAC + capacity, plus **JWT validation** for Foundry contracts)
- Foundry connection name(s) printed explicitly for use in downstream notebooks
  (e.g. `citadel-agent-frameworks-tests.ipynb`)
- Validated request/response behaviour, throttling and token-bucket visualisation

## Azure Prerequisites

To take full advantage of this notebook, ensure you have the following Azure resources set up:
- An existing Citadel Governance Hub deployment (APIM + supporting resources)
- *(Optional)* An Azure Key Vault with secrets for API keys (if testing Key Vault integration)
- *(Optional)* A Microsoft Foundry account & project (if testing Foundry connection integration). For the
  `ProjectManagedIdentity` default, enable a managed identity on the Foundry **project** (Project → Identity)
- Azure credentials with permissions to deploy at subscription scope and access the above resources

> **Note:** This notebook assumes you have already deployed your Citadel Governance Hub. If you haven't done so, please refer to the [Citadel Access Contracts Guide](../guides/full-deployment-guide.md) before proceeding.


<a id='0'></a>
### 0️⃣ Initialize Notebook Variables

**Choose ONE initialization mode** by setting `init_from_azd`:

- `True` — autoload `governance_hub_resource_group`, `location`, `keyvault_name`, and the Foundry account/project from your active `azd` environment (`azd env get-value ...`). Works when the accelerator was deployed with `azd up`.
- `False` — fill the `REPLACE` values manually below.

In [ ]:
import os
import sys, json, requests, time
sys.path.insert(1, '../shared')  # add the shared directory to the Python path
import utils
from apimtools import APIMClientTool

# ============================================================================
# 🔧 INITIALIZATION MODE
# ============================================================================
init_from_azd = True   # Set False to fill the REPLACE values below manually.

# ============================================================================
# 🔧 GOVERNANCE HUB CONFIGURATION (REQUIRED — used as defaults if azd lookup fails)
# ============================================================================
governance_hub_resource_group = "REPLACE"   # Resource group of the deployed Citadel Governance Hub
location = "REPLACE"                         # Azure region (e.g. "swedencentral", "eastus")

# ============================================================================
# 🔧 API VERSION CONFIGURATION
# ============================================================================
inference_api_version = "2024-05-01-preview"
openai_api_version    = "2024-12-01-preview"
targetInferenceApi    = "models"             # 'models' = Universal LLM API | 'openai' = Azure OpenAI API

# ============================================================================
# 🔐 KEY VAULT CONFIGURATION (optional - set use_keyvault_integration = True to enable)
# ----------------------------------------------------------------------------
# Any value left as "REPLACE" (or empty) will be filled in from azd when
# `init_from_azd = True`. Any value you set here explicitly will be honored
# as-is and will NOT be overridden by azd. This lets you point the contract
# at a different Key Vault (e.g. a shared central vault) than the one azd
# provisioned with the hub.
# ============================================================================
use_keyvault_integration = True
keyvault_subscription_id = "d2e7f84f-2790-4baa-9520-59ae8169ed0d"
keyvault_resource_group  = "rg-citadel-agent-09"
keyvault_name            = "kv-citadel-agent-09"

# ============================================================================
# 🤖 MICROSOFT FOUNDRY CONFIGURATION (optional - set use_foundry_integration = True to enable)
# Same precedence rule as Key Vault: explicit values win, "REPLACE" is filled by azd.
# ============================================================================
use_foundry_integration = True
foundry_subscription_id = "d2e7f84f-2790-4baa-9520-59ae8169ed0d"
foundry_resource_group  = "rg-citadel-agent-09"
foundry_account_name    = "aif-citadel-agent-09"
foundry_project_name    = "proj-citadel-agent-09"

# ============================================================================
# 🔁 azd ENVIRONMENT OVERRIDES
# ----------------------------------------------------------------------------
# When `init_from_azd = True` we pull the deployment-time outputs from your
# active azd environment (set by `azd up`). User-supplied values above (i.e.
# anything that is NOT "REPLACE" or empty) ALWAYS take precedence — azd is
# only consulted to fill in the gaps.
# ============================================================================

def _is_unset(value):
    """A value counts as 'not provided' when the user left it as REPLACE or blank."""
    return value is None or value == "" or value == "REPLACE"


if init_from_azd:
    utils.print_info("Loading configuration from azd environment...")
    loaded = utils.load_azd_env({
        "resource_group":      ["AZURE_RESOURCE_GROUP", "GOVERNANCE_HUB_RESOURCE_GROUP"],
        "location":            ["AZURE_LOCATION", "LOCATION"],
        "subscription_id":     ["AZURE_SUBSCRIPTION_ID"],
        "key_vault_name":      ["KEY_VAULT_NAME"],
        "ai_foundry_services": (["AI_FOUNDRY_SERVICES"], "json"),
    }, verbose=False)

    if _is_unset(governance_hub_resource_group) and "resource_group" in loaded:
        governance_hub_resource_group = loaded["resource_group"]
    if _is_unset(location) and "location" in loaded:
        location = loaded["location"]

    # ---- Key Vault: fill any unset field; never override user-supplied values ----
    if _is_unset(keyvault_name) and "key_vault_name" in loaded:
        keyvault_name = loaded["key_vault_name"]
    if _is_unset(keyvault_subscription_id) and "subscription_id" in loaded:
        keyvault_subscription_id = loaded["subscription_id"]
    if _is_unset(keyvault_resource_group) and not _is_unset(governance_hub_resource_group):
        # Default the Key Vault resource group to the hub's RG only when the user didn't pin one.
        keyvault_resource_group = governance_hub_resource_group

    # ---- Foundry: fill any unset field; never override user-supplied values ----
    if "ai_foundry_services" in loaded and isinstance(loaded["ai_foundry_services"], list) and loaded["ai_foundry_services"]:
        first = loaded["ai_foundry_services"][0]
        if _is_unset(foundry_account_name):
            foundry_account_name = first.get("cognitiveServiceName") or first.get("name") or foundry_account_name
        if _is_unset(foundry_project_name):
            ep = first.get("foundryProjectEndpoint", "")
            # Endpoint shape: https://<account>.services.ai.azure.com/api/projects/<projectName>
            if "/projects/" in ep:
                foundry_project_name = ep.rstrip("/").rsplit("/projects/", 1)[-1]
        if _is_unset(foundry_subscription_id) and "subscription_id" in loaded:
            foundry_subscription_id = loaded["subscription_id"]
        if _is_unset(foundry_resource_group) and not _is_unset(governance_hub_resource_group):
            foundry_resource_group = governance_hub_resource_group

    utils.print_ok(f"Resource group : {governance_hub_resource_group}")
    utils.print_ok(f"Location       : {location}")
    utils.print_ok(f"Key Vault      : {keyvault_name} (rg={keyvault_resource_group}, sub={keyvault_subscription_id})")
    utils.print_ok(f"Foundry        : {foundry_account_name} / {foundry_project_name}")

utils.print_ok("Notebook variables initialized!")


<a id='1'></a>
### 1️⃣ Verify Azure CLI and Connected Subscription

Ensure Azure CLI is authenticated and connected to the correct subscription:

In [ ]:
output = utils.run("az account show", "Retrieved az account", "Failed to get the current az account")

if output.success and output.json_data:
    current_user = output.json_data['user']['name']
    tenant_id = output.json_data['tenantId']
    subscription_id = output.json_data['id']

    utils.print_info(f"Current user: {current_user}")
    utils.print_info(f"Tenant ID: {tenant_id}")
    utils.print_info(f"Subscription ID: {subscription_id}")

<a id='2'></a>
### 2️⃣ Initialize APIM Client Tool

👉 An existing Citadel Governance Hub deployment is expected. Initialize the APIM client to interact with your deployment:

In [ ]:
try:
    apimClientTool = APIMClientTool(
        governance_hub_resource_group
    )
    apimClientTool.initialize()
    apimClientTool.discover_api(targetInferenceApi)

    apim_resource_gateway_url = str(apimClientTool.apim_resource_gateway_url)
    azure_endpoint = str(apimClientTool.azure_endpoint)
    
    # Get supported models from the policy fragment
    supported_models = apimClientTool.get_policy_fragment_supported_models("set-backend-pools")
    utils.print_info(f"Supported models in APIM policy fragment 'set-backend-pools': {supported_models}")

    if targetInferenceApi == "openai":
        chat_completions_url = f"{azure_endpoint}openai/deployments/{{model_name}}/chat/completions?api-version={inference_api_version}"
    else:  # models
        chat_completions_url = f"{azure_endpoint}models/chat/completions?api-version={inference_api_version}"
    utils.print_info(f"Chat Completion Endpoint Template: {chat_completions_url}")

    utils.print_info(f"Using the following API: {apimClientTool.api_id}")

    utils.print_ok(f"Testing tool initialized successfully!")
except Exception as e:
    utils.print_error(f"Error initializing APIM Client Tool: {e}")

<a id='3'></a>
### 3️⃣ Define Access Contract Configurations

We will create 3 different access contracts with varying default configurations:
1. **Sales-Assistant**: Key Vault integration only
2. **HR-ChatAgent**: Key Vault + Foundry connection integrations (if enabled)
3. **Support-Bot**: Direct output (no Key Vault nor Foundry integration)

> 🔐 **Foundry connection auth (new default):** contracts that target Foundry (`use_foundry = True`,
> e.g. **HR-ChatAgent**) now create the connection with **`authType = ProjectManagedIdentity`**. The
> Foundry project's managed identity presents an Entra ID **Bearer token** (audience
> `https://cognitiveservices.azure.com`) while the **APIM subscription key** rides along as the
> `api-key` **custom header**. The generated product policy therefore enables the gateway's JWT
> validation for that audience (reusing the existing `security-handler`), so **both** the api-key and
> the JWT must be valid. Set `authType = 'ApiKey'` on `foundryConfig` to fall back to the original
> subscription-key-only behavior (backward compatible).


In [ ]:
# Define the access contracts to create
# Each contract supports per-contract:
#   • allowed_models: list of model names this contract is allowed to call (model-level RBAC)
#   • tokens_per_minute: TPM throttle limit applied to the contract
#   • token_quota / token_quota_period: long-term token budget (Hourly | Daily | Weekly | Monthly)
# These values are injected into a dynamically-generated APIM product policy
# (see next cell) so each contract gets its own scoped policy XML.

timestamp = time.strftime('%Y%m%d%H%M%S')

access_contracts = [
    {
        "name": f"sales-assistant-contract-{timestamp}",
        "business_unit": "Sales",
        "use_case_name": "Assistant",
        "environment": "DEV",
        "use_keyvault": True,
        "use_foundry": False,
        "endpoint_secret": "SALES-LLM-ENDPOINT",
        "apikey_secret": "SALES-LLM-KEY",
        "description": "Sales Assistant - Key Vault only",
        # --- Model RBAC (allowed model names, comma-joined into the APIM policy)
        "allowed_models": ["gpt-4.1", "gpt-5.4-mini", "text-embedding-3-large"],
        # --- Capacity allocation (llm-token-limit policy)
        "tokens_per_minute": 3000,
        "token_quota": 100000,
        "token_quota_period": "Monthly"
    },
    {
        "name": f"hr-chatagent-contract-{timestamp}",
        "business_unit": "HR",
        "use_case_name": "ChatAgent",
        "environment": "DEV",
        "use_keyvault": True,
        "use_foundry": True,
        "endpoint_secret": "HR-LLM-ENDPOINT",
        "apikey_secret": "HR-LLM-KEY",
        "description": "HR Chat Agent - Key Vault + Foundry (if enabled)",
        "allowed_models": ["gpt-4.1", "gpt-5.4-mini"],
        "tokens_per_minute": 5000,
        "token_quota": 500000,
        "token_quota_period": "Monthly"
    },
    {
        "name": f"support-bot-contract-{timestamp}",
        "business_unit": "Support",
        "use_case_name": "Bot",
        "environment": "DEV",
        "use_keyvault": False,
        "use_foundry": False,
        "endpoint_secret": "SUPPORT-LLM-ENDPOINT",
        "apikey_secret": "SUPPORT-LLM-KEY",
        "description": "Support Bot - Direct output (no Key Vault nor Foundry connection integration)",
        "allowed_models": ["gpt-4.1","Mistral-Large-3"],
        "tokens_per_minute": 2000,
        "token_quota": 50000,
        "token_quota_period": "Daily"
    }
]

utils.print_info(f"Defined {len(access_contracts)} access contracts to create:")
for i, contract in enumerate(access_contracts, 1):
    utils.print_info(f"  {i}. {contract['description']}")
    utils.print_info(f"     Product ID: LLM-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}")
    utils.print_info(f"     Allowed models: {', '.join(contract['allowed_models'])}")
    utils.print_info(f"     Capacity: {contract['tokens_per_minute']} TPM, quota {contract['token_quota']} / {contract['token_quota_period']}")


<a id='4'></a>
### 4️⃣ Create Access Contract Parameter Files

Generate Bicep parameter files (`.bicepparam`) for each access contract.
These files configure the APIM products, subscriptions, and optionally Key Vault secrets and Foundry connections.

In [ ]:
import shutil

bicep_dir = "../bicep/infra/citadel-access-contracts"
template_file = os.path.join(bicep_dir, "main.bicep")

# Audience the Foundry project managed identity requests a token for (and that the product
# policy validates). This is the default for Cognitive Services / Azure AI gateway connections.
foundry_mi_audience = "https://cognitiveservices.azure.com"

# Store generated parameter files for deployment
generated_param_files = []

def build_foundry_jwt_policy(contract):
    """When a contract targets Foundry with the new default 'ProjectManagedIdentity' connection,
    the project managed identity presents an Entra ID Bearer token (audience = cognitive services)
    ALONGSIDE the api-key custom header. The product policy therefore enables the existing JWT
    validation (security-handler) scoped to the cognitive services audience. The tenant id is baked
    in so the generated policy is self-contained (no dependency on the JWT-TenantId named value)."""
    if not contract.get("use_foundry"):
        return ""
    return f'''
        <!-- Foundry ProjectManagedIdentity: require + validate the Entra ID Bearer token issued to
             the project managed identity for the cognitive services audience, IN ADDITION to the
             api-key subscription key. Reuses the gateway's security-handler JWT validation. -->
        <set-variable name="jwtRequired" value="true" />
        <set-variable name="jwtAudience" value="{foundry_mi_audience}" />
        <set-variable name="jwtIssuer" value="https://sts.windows.net/{tenant_id}/" />
        <set-variable name="jwtOpenIdConfigUrl" value="https://login.microsoftonline.com/{tenant_id}/v2.0/.well-known/openid-configuration" />
'''

def build_product_policy_xml(contract):
    """Generate a per-contract APIM product policy XML applying:
       - Model-level RBAC via the `allowedModels` variable + validate-model-access fragment
       - Capacity allocation via llm-token-limit (tokens-per-minute + token-quota)
       - (Foundry contracts only) JWT validation for the ProjectManagedIdentity Bearer token
    Mirrors the dynamic-policy approach used in citadel-unified-ai-api-tests.ipynb."""
    allowed_csv = ",".join(contract["allowed_models"])
    return f'''<policies>
    <inbound>
        <base />
{build_foundry_jwt_policy(contract)}
        <!-- Extract and validate model parameter from request -->
        <include-fragment fragment-id="set-llm-requested-model" />

        <!-- Model-level RBAC: only the models below are allowed for this contract -->
        <set-variable name="allowedModels" value="{allowed_csv}" />
        <include-fragment fragment-id="validate-model-access" />

        <!-- Capacity allocation: per-subscription token throttling + long-term quota -->
        <llm-token-limit counter-key="@(context.Subscription.Id)"
                         tokens-per-minute="{contract["tokens_per_minute"]}"
                         estimate-prompt-tokens="false"
                         token-quota="{contract["token_quota"]}"
                         token-quota-period="{contract["token_quota_period"]}" />

        <!-- Enable advanced response headers offered by set-response-headers fragment -->
        <set-variable name="enableResponseHeaders" value="@(true)" />
    </inbound>
    <backend>
        <base />
    </backend>
    <outbound>
        <base />
    </outbound>
    <on-error>
        <base />
    </on-error>
</policies>'''


for i, contract in enumerate(access_contracts, 1):
    utils.print_info(f"\n{'='*60}")
    utils.print_info(f"Creating Parameter File {i}/{len(access_contracts)}: {contract['description']}")
    utils.print_info(f"{'='*60}")

    # Folder structure: contracts/[businessunit-usecase]/[environment]/
    folder_name = f"{contract['business_unit'].lower()}-{contract['use_case_name'].lower()}"
    environment_folder = contract['environment'].lower()
    contract_folder = os.path.join(bicep_dir, "contracts", folder_name, environment_folder)
    os.makedirs(contract_folder, exist_ok=True)
    utils.print_info(f"📁 Created folder: {contract_folder}")

    # Generate per-contract policy XML (model RBAC + capacity + optional Foundry MI JWT)
    policy_xml = build_product_policy_xml(contract)
    policy_file_dest = os.path.join(contract_folder, "ai-product-policy.xml")
    with open(policy_file_dest, "w") as f:
        f.write(policy_xml)
    utils.print_ok(f"📋 Generated dynamic policy: {policy_file_dest}")
    utils.print_info(f"     • Allowed models : {', '.join(contract['allowed_models'])}")
    utils.print_info(f"     • TPM            : {contract['tokens_per_minute']}")
    utils.print_info(f"     • Token quota    : {contract['token_quota']} / {contract['token_quota_period']}")
    if contract.get('use_foundry'):
        utils.print_info(f"     • Foundry auth   : ProjectManagedIdentity + api-key header (JWT audience {foundry_mi_audience})")

    # Path used by Bicep loadTextContent (relative to the .bicepparam file)
    policy_relative_path = "ai-product-policy.xml"
    params_file = os.path.join(contract_folder, "main.bicepparam")

    # Build Foundry configuration section
    if contract['use_foundry']:
        foundry_params = f"""
// Azure AI Foundry Integration
// Default connection auth = ProjectManagedIdentity: the project managed identity presents an Entra ID
// Bearer token (audience below) AND the subscription key is sent as the api-key custom header.
// The product policy validates that JWT (see ai-product-policy.xml). Set authType to 'ApiKey' to fall
// back to the original subscription-key-only behavior.
param useTargetFoundry = true

param foundry = {{
  subscriptionId: '{foundry_subscription_id}'
  resourceGroupName: '{foundry_resource_group}'
  accountName: '{foundry_account_name}'
  projectName: '{foundry_project_name}'
}}

param foundryConfig = {{
  connectionNamePrefix: ''
  authType: 'ProjectManagedIdentity'
  managedIdentityAudience: '{foundry_mi_audience}'
  deploymentInPath: 'false'
  isSharedToAll: false
  inferenceAPIVersion: '2024-05-01-preview'
  deploymentAPIVersion: ''
  staticModels: [
    {{
      name: 'gpt-4.1'
      properties: {{
        model: {{
          name: 'gpt-4.1'
          version: '2025-04-14'
          format: 'OpenAI'
        }}
      }}
    }}
    {{
      name: 'gpt-5.4-mini'
      properties: {{
        model: {{
          name: 'gpt-5.4-mini'
          version: '2025-04-14'
          format: 'OpenAI'
        }}
      }}
    }}
  ]
  listModelsEndpoint: ''
  getModelEndpoint: ''
  deploymentProvider: ''
  customHeaders: {{}}
  authConfig: {{}}
}}
"""
    else:
        foundry_params = """
// Azure AI Foundry Integration (disabled)
param useTargetFoundry = false

param foundry = {
  subscriptionId: '00000000-0000-0000-0000-000000000000'
  resourceGroupName: 'placeholder'
  accountName: 'placeholder'
  projectName: 'placeholder'
}
"""

    params_content = f"""using '../../../main.bicep'

// ============================================================================
// {contract['description']} - Generated from Notebook
// Dynamic policy attributes:
//   allowedModels       : {','.join(contract['allowed_models'])}
//   tokens-per-minute   : {contract['tokens_per_minute']}
//   token-quota         : {contract['token_quota']} / {contract['token_quota_period']}
// ============================================================================

param apim = {{
  subscriptionId: '{subscription_id}'
  resourceGroupName: '{governance_hub_resource_group}'
  name: '{apimClientTool.apim_resource_name}'
}}

param keyVault = {{
  subscriptionId: '{keyvault_subscription_id}'
  resourceGroupName: '{keyvault_resource_group}'
  name: '{keyvault_name}'
}}

param useTargetAzureKeyVault = {str(contract['use_keyvault']).lower()}

param useCase = {{
  businessUnit: '{contract['business_unit']}'
  useCaseName: '{contract['use_case_name']}'
  environment: '{contract['environment']}'
}}

param apiNameMapping = {{
  LLM: ['universal-llm-api', 'azure-openai-api', 'unified-ai-api']
}}

param services = [
  {{
    code: 'LLM'
    endpointSecretName: '{contract['endpoint_secret']}'
    apiKeySecretName: '{contract['apikey_secret']}'
    policyXml: loadTextContent('{policy_relative_path}')
  }}
]

param productTerms = 'Access Contract created from testing notebook - {contract["description"]}'
{foundry_params}
"""

    with open(params_file, "w") as f:
        f.write(params_content)

    generated_param_files.append({
        "contract": contract,
        "params_file": params_file,
        "contract_folder": contract_folder
    })
    utils.print_ok(f"✅ Parameter file created: {params_file}")

utils.print_ok(f"\n📁 Created {len(generated_param_files)} parameter files ready for deployment!")
utils.print_info("Each contract folder contains:")
utils.print_info("  • main.bicepparam - Bicep parameter file")
utils.print_info("  • ai-product-policy.xml - Dynamically generated APIM product policy (model RBAC + capacity)")


<a id='4.1'></a>
### 4️⃣.1 Deploy Access Contracts Using 🦾 Bicep

Deploy each access contract using the generated parameter files.
This creates the APIM products, subscriptions, and optionally Key Vault secrets and Foundry connections in Azure.

In [ ]:
# Store deployment results for later use
deployment_results = []

for i, item in enumerate(generated_param_files, 1):
    contract = item['contract']
    params_file = item['params_file']
    
    utils.print_info(f"\n{'='*60}")
    utils.print_info(f"Deploying Access Contract {i}/{len(generated_param_files)}: {contract['description']}")
    utils.print_info(f"{'='*60}")
    
    # Deploy the access contract
    deployment_cmd = f"az deployment sub create --name {contract['name']} --location {location} --template-file {template_file} --parameters {params_file}"
    
    utils.print_info(f"Deploying {contract['name']}...")
    output = utils.run(
        deployment_cmd,
        f"Deployment '{contract['name']}' succeeded",
        f"Deployment '{contract['name']}' failed"
    )

    if output.success:
        # Deployment succeeded - try to get outputs if JSON data is available
        outputs = {}
        if output.json_data:
            outputs = output.json_data.get('properties', {}).get('outputs', {})
        
        deployment_results.append({
            "contract": contract,
            "outputs": outputs,
            "success": True
        })
        utils.print_ok(f"✅ Access Contract {i} deployed successfully!")
        
        # Show key outputs if available
        if outputs:
            for key, value in outputs.items():
                masked_value = utils.mask_sensitive_values(value.get('value'))
                utils.print_info(f"  {key}: {masked_value}")
        else:
            utils.print_info("  (No outputs returned - deployment completed)")
    else:
        deployment_results.append({
            "contract": contract,
            "outputs": {},
            "success": False
        })
        utils.print_error(f"❌ Access Contract {i} deployment failed!")

# Re-initialize APIM client to pick up new subscriptions
apimClientTool.initialize()
utils.print_ok(f"\n🎉 Completed deploying {len([r for r in deployment_results if r['success']])} access contracts!")

<a id='4.2'></a>
### 4️⃣.2 Foundry Connection Names (for downstream notebooks)

When `use_foundry = True` for an access contract, the deployment provisions a Microsoft Foundry
project connection that points back to the APIM gateway. **This connection name is what other
notebooks (e.g., `citadel-agent-frameworks-tests.ipynb`) use as `foundry_connection_name`.**

> 🔐 **Connection auth = `ProjectManagedIdentity` (new default).** The connection is created with
> `authType = ProjectManagedIdentity` and the APIM subscription key stored as the `api-key`
> **custom header**. At runtime the Foundry project managed identity acquires an Entra ID token for
> the audience `https://cognitiveservices.azure.com` and sends it as `Authorization: Bearer …`,
> while the `api-key` header carries the subscription key. **Prerequisite:** enable a system- or
> user-assigned managed identity on the Foundry **project** (Project → Identity) and grant it access;
> the gateway validates the token via the access contract's product policy. Use
> `authType = 'ApiKey'` to revert to subscription-key-only auth.

**Naming convention** (see `bicep/infra/citadel-access-contracts/main.bicep`):

```
<connectionNamePrefix>-<ServiceCode>
```

Where `connectionNamePrefix` defaults to `Hub-<businessUnit>-<useCaseName>-<environment>` when left
empty in `foundryConfig` (which is the default in this notebook).

So the resulting connection names follow the pattern:

```
Hub-<BusinessUnit>-<UseCaseName>-<Environment>-<ServiceCode>      # e.g. Hub-HR-ChatAgent-DEV-LLM
```

The next cell extracts the connection name(s) from the deployment outputs and prints them
explicitly so you can copy them into other notebooks.


In [ ]:
# Extract and display the Foundry connection name(s) for each deployed access contract.
# Use these values as `foundry_connection_name` in the agent-frameworks notebook.

foundry_connection_names = {}

print("\n" + "=" * 70)
print("🔗 FOUNDRY CONNECTION NAMES (for downstream notebooks)")
print("=" * 70)
print("Naming convention: <BusinessUnit>-<UseCaseName>-<Environment>-<ServiceCode>")
print("                   (when foundryConfig.connectionNamePrefix is empty)")
print("-" * 70)

for result in deployment_results:
    contract = result["contract"]
    if not contract.get("use_foundry"):
        continue
    if not result.get("success"):
        utils.print_warning(f"⚠️  Skipping {contract['name']} (deployment failed)")
        continue

    # Prefer the value emitted by the Bicep `foundryConnections` output
    outputs = result.get("outputs") or {}
    fc_output = outputs.get("foundryConnections", {}).get("value", [])
    if fc_output:
        for entry in fc_output:
            conn_name = entry.get("connectionName") if isinstance(entry, dict) else entry
            if conn_name:
                foundry_connection_names.setdefault(contract["name"], []).append(conn_name)
    else:
        # Fallback: reconstruct from the naming convention
        for svc_code in ["LLM"]:
            conn_name = f"{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}-{svc_code}"
            foundry_connection_names.setdefault(contract["name"], []).append(conn_name)

    for cn in foundry_connection_names.get(contract["name"], []):
        utils.print_ok(f"✅ {contract['description']}")
        print(f"     foundry_connection_name = \"{cn}\"")

if not foundry_connection_names:
    utils.print_info("No Foundry-enabled contracts in this run.")
else:
    print("-" * 70)
    print("📋 Copy/paste-ready map:")
    print(json.dumps(foundry_connection_names, indent=2))
print("=" * 70)


<a id='5'></a>
### 5️⃣ Retrieve API Keys for Each Access Contract

Get the subscription keys created for each access contract to use in API testing.

In [ ]:
# Map contract names to their subscription keys
contract_keys = {}

for result in deployment_results:
    if not result['success']:
        continue
    
    contract = result['contract']
    product_id = f"LLM-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}"
    subscription_name = f"{product_id}-SUB-01"
    
    # Find the subscription key from APIM subscriptions
    for sub in apimClientTool.apim_subscriptions:
        if subscription_name.lower() in sub.get('name', '').lower():
            contract_keys[product_id] = {
                "key": sub.get('key'),
                "description": contract['description'],
                "use_keyvault": contract['use_keyvault'],
                "use_foundry": contract['use_foundry']
            }
            utils.print_ok(f"Found key for {product_id}")
            break
    else:
        # If not found in existing subscriptions, check outputs for direct credentials
        if not contract['use_keyvault']:
            endpoints = result['outputs'].get('endpoints', {}).get('value', [])
            for ep in endpoints:
                if ep.get('code') == 'LLM':
                    contract_keys[product_id] = {
                        "key": ep.get('apiKey'),
                        "endpoint": ep.get('endpoint'),
                        "description": contract['description'],
                        "use_keyvault": contract['use_keyvault'],
                        "use_foundry": contract['use_foundry']
                    }
                    utils.print_ok(f"Found direct key for {product_id}")
                    break

utils.print_info(f"\nRetrieved keys for {len(contract_keys)} access contracts:")
for product_id, info in contract_keys.items():
    utils.print_info(f"  • {product_id}: {info['description']}")

# ============================================================================
# 🔐 Foundry (ProjectManagedIdentity) contracts also require a JWT Bearer token
# ----------------------------------------------------------------------------
# Foundry-targeted contracts enable JWT validation for the cognitive services audience
# (the token a project managed identity would present). The product policy requires BOTH the
# api-key AND a valid Bearer token. From this notebook (running under `az login`) we mint an
# equivalent Entra ID token for the SAME audience so the direct HTTP tests below succeed — this
# exercises the exact validation the Foundry project MID token goes through at runtime.
# ============================================================================
foundry_bearer_token = None
if any(info.get("use_foundry") for info in contract_keys.values()):
    token_out = utils.run(
        f"az account get-access-token --resource {foundry_mi_audience}",
        "Acquired Entra ID token for the Foundry managed-identity audience",
        "Failed to acquire Entra ID token for the Foundry managed-identity audience"
    )
    if token_out.success and token_out.json_data:
        foundry_bearer_token = token_out.json_data.get("accessToken")
        utils.print_ok(f"🔐 Bearer token ready for audience: {foundry_mi_audience}")
    else:
        utils.print_warning("⚠️ Could not acquire a Bearer token; Foundry (JWT-enabled) contract tests will return 401.")


def build_contract_headers(info):
    """Build request headers for a contract test call.
    Always includes the api-key. For Foundry (ProjectManagedIdentity) contracts the JWT is also
    required, so an Authorization: Bearer token (cognitive services audience) is attached."""
    headers = {"api-key": info.get("key")}
    if info.get("use_foundry") and foundry_bearer_token:
        headers["Authorization"] = f"Bearer {foundry_bearer_token}"
    return headers


<a id='6'></a>
### 6️⃣ Test API Requests Across All Access Contracts

Send test requests to each access contract and collect metrics for visualization.

In [ ]:
model_name = "gpt-5.4-mini"
# if you want to dynamically select model from supported models, uncomment below
# model_name = supported_models[2] if len(supported_models) > 2 else supported_models[0]
utils.print_info(f"Using model: {model_name}")

# Store results for each contract
test_results = {product_id: [] for product_id in contract_keys.keys()}

messages = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant. Keep responses brief."},
        {"role": "user", "content": "What is 2+2?"}
    ]
}

# Send a single test request to each contract
for product_id, info in contract_keys.items():
    utils.print_info(f"\nTesting {product_id}...")

    api_key = info.get('key')
    if not api_key:
        utils.print_error(f"No API key found for {product_id}")
        continue

    # Foundry contracts require api-key + JWT Bearer (ProjectManagedIdentity audience)
    headers = build_contract_headers(info)
    if info.get('use_foundry'):
        auth_mode = "api-key + JWT Bearer" if "Authorization" in headers else "api-key only (⚠️ JWT missing)"
        utils.print_info(f"   Auth: {auth_mode} (ProjectManagedIdentity contract)")

    try:
        response = requests.post(
            chat_completions_url,
            headers=headers,
            json=messages,
            timeout=30
        )
        
        utils.print_response_code(response)
        
        if response.status_code == 200:
            data = json.loads(response.text)
            content = data.get("choices", [{}])[0].get("message", {}).get("content", "")
            utils.print_ok(f"💬 Response: {content[:100]}..." if len(content) > 100 else f"💬 Response: {content}")
            utils.print_info(f"   Region: {response.headers.get('x-ms-region', 'N/A')}")
        else:
            utils.print_error(f"Error: {response.text[:350]}")
    except Exception as e:
        utils.print_error(f"Request failed: {e}")


<a id='7'></a>
### 7️⃣ Run Load Test Across All Access Contracts

Send multiple requests to each contract over 30 seconds to test rate limiting and collect performance data.

In [ ]:
import requests, json, time
from concurrent.futures import ThreadPoolExecutor
import threading

# Run for 30 seconds per contract
test_duration = 30
all_api_runs = {product_id: [] for product_id in contract_keys.keys()}

model_name = "gpt-4.1" # This is a common model that is allowed by default in the contracts, but you can change it as needed
utils.print_info(f"Using model: {model_name}")

messages = {
    "model": model_name,
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Count from 1 to 10."}
    ]
}

def run_api_test(product_id, headers, duration):
    """Run API calls for a specific contract using the provided headers
    (api-key, plus Authorization Bearer for Foundry ProjectManagedIdentity contracts)."""
    runs = []
    start_time = time.time()
    run_count = 0
    
    while (time.time() - start_time) < duration:
        run_count += 1
        call_start_time = time.time()
        
        try:
            response = requests.post(
                chat_completions_url,
                headers=headers,
                json=messages,
                timeout=30
            )
            
            elapsed = time.time() - start_time
            
            if response.status_code == 200:
                data = json.loads(response.text)
                total_tokens = data.get("usage", {}).get("total_tokens", 0)
            else:
                total_tokens = 0
            
            runs.append((call_start_time, total_tokens, response.status_code, elapsed))
            
        except Exception as e:
            runs.append((call_start_time, 0, 500, time.time() - start_time))
        
        time.sleep(0.2)  # Small delay between requests
    
    return runs

# Run tests for each contract sequentially
for product_id, info in contract_keys.items():
    api_key = info.get('key')
    if not api_key:
        continue

    headers = build_contract_headers(info)
    auth_note = " (api-key + JWT Bearer)" if info.get('use_foundry') and "Authorization" in headers else ""

    print(f"\n🕐 Testing {product_id} for {test_duration} seconds...")
    print(f"   {info['description']}{auth_note}")

    runs = run_api_test(product_id, headers, test_duration)
    all_api_runs[product_id] = runs
    
    success = sum(1 for r in runs if r[2] == 200)
    throttled = sum(1 for r in runs if r[2] == 429)
    errors = sum(1 for r in runs if r[2] not in [200, 429])
    
    print(f"   ✅ Success: {success} | ⛔ Throttled: {throttled} | ❌ Errors: {errors}")

utils.print_ok(f"\n🏁 Load testing completed for all {len(contract_keys)} access contracts!")


<a id='8'></a>
### 8️⃣ Visualize Results Across All Access Contracts

Compare API usage, token consumption, and throttling behavior across all access contracts.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import numpy as np

# Check if we have data to plot
contracts_with_data = {k: v for k, v in all_api_runs.items() if v}

if contracts_with_data:
    # Print summary table
    print("\n" + "="*80)
    print("📊 SUMMARY: Access Contracts Performance Comparison")
    print("="*80)
    print(f"{'Contract':<40} {'Calls':<8} {'Success':<10} {'Throttled':<10} {'Tokens':<10}")
    print("-"*80)
    
    for product_id, runs in contracts_with_data.items():
        success = sum(1 for r in runs if r[2] == 200)
        throttled = sum(1 for r in runs if r[2] == 429)
        total_tokens = sum(r[1] for r in runs)
        print(f"{product_id:<40} {len(runs):<8} {success:<10} {throttled:<10} {total_tokens:<10}")
    
    print("="*80)
    
    num_contracts = len(contracts_with_data)
    fig, axes = plt.subplots(num_contracts, 1, figsize=(14, 5 * num_contracts), squeeze=False)
    
    colors_map = {'success': 'tab:green', 'throttled': 'tab:red', 'error': 'tab:orange'}
    
    for idx, (product_id, runs) in enumerate(contracts_with_data.items()):
        ax = axes[idx, 0]
        
        if not runs:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center')
            ax.set_title(product_id)
            continue
        
        # Process data
        base_time = runs[0][0]
        times = [r[3] for r in runs]  # elapsed time
        tokens = [r[1] for r in runs]
        status_codes = [r[2] for r in runs]
        
        # Color bars based on status
        colors = [
            colors_map['success'] if code == 200 
            else colors_map['throttled'] if code == 429 
            else colors_map['error'] 
            for code in status_codes
        ]
        
        # Create bar chart
        ax.bar(times, tokens, color=colors, width=0.3, alpha=0.7)
        
        # Add throttled markers
        throttled_times = [t for t, code in zip(times, status_codes) if code == 429]
        if throttled_times:
            max_tokens = max(tokens) if tokens else 1
            ax.scatter(throttled_times, [max_tokens * 0.05] * len(throttled_times), 
                      marker='x', s=50, color='darkred', zorder=5)
        
        # Calculate stats
        success = sum(1 for code in status_codes if code == 200)
        throttled = sum(1 for code in status_codes if code == 429)
        total_tokens = sum(tokens)
        
        # Labels and title
        info = contract_keys.get(product_id, {})
        title = f"{product_id}\n{info.get('description', '')}"
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (seconds)')
        ax.set_ylabel('Tokens per call')
        
        # Add stats annotation
        stats_text = f"Total: {len(runs)} calls | Success: {success} | Throttled: {throttled} | Tokens: {total_tokens}"
        ax.text(0.02, 0.98, stats_text, transform=ax.transAxes, fontsize=9,
               verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Add shared legend
    legend_items = [
        Patch(facecolor='tab:green', alpha=0.7, label='Success (200)'),
        Patch(facecolor='tab:red', alpha=0.7, label='Throttled (429)'),
        Patch(facecolor='tab:orange', alpha=0.7, label='Error'),
        Line2D([0], [0], marker='x', color='darkred', markersize=8, linestyle='None', label='Throttle point')
    ]
    fig.legend(handles=legend_items, loc='upper right', bbox_to_anchor=(0.98, 0.99))
    
    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    plt.show()
    
else:
    print('No API test data available. Run the load test first to capture data.')

<a id='9'></a>
### 9️⃣ Compare Token Bucket Behavior

Visualize the token bucket algorithm behavior for each access contract.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

contracts_with_data = {k: v for k, v in all_api_runs.items() if v}

if contracts_with_data:
    # Token bucket parameters (default policy: 400 tokens/min)
    capacity = 400
    refill = capacity / 60  # tokens per second
    
    fig, axes = plt.subplots(len(contracts_with_data), 1, figsize=(14, 6 * len(contracts_with_data)), squeeze=False)
    
    for idx, (product_id, runs) in enumerate(contracts_with_data.items()):
        ax1 = axes[idx, 0]
        ax2 = ax1.twinx()
        
        # Process data for token bucket simulation
        calls = [(r[3], r[1] or 0, r[2]) for r in runs]  # (elapsed_time, tokens, status)
        
        bucket = capacity
        last_time = 0.0
        times, usage, status_codes, levels = [], [], [], []
        
        for call_time, tokens, status in calls:
            # Refill bucket
            bucket = min(capacity, bucket + (call_time - last_time) * refill)
            levels.append(bucket)
            times.append(call_time)
            usage.append(tokens)
            status_codes.append(status)
            # Consume tokens
            bucket = max(0, bucket - tokens)
            last_time = call_time
        
        # Colors based on status
        colors = ['tab:green' if code == 200 else 'tab:red' if code == 429 else 'tab:orange' for code in status_codes]
        
        # Plot bars for token usage
        ax1.bar(times, usage, color=colors, width=0.35, alpha=0.7)
        
        # Plot bucket level
        ax2.plot(times, levels, color='purple', linewidth=2)
        ax2.axhline(capacity, color='purple', linestyle='--', alpha=0.6)
        
        # Mark throttled points
        throttled_times = [t for t, code in zip(times, status_codes) if code == 429]
        throttled_usage = [u for u, code in zip(usage, status_codes) if code == 429]
        if throttled_times:
            max_usage = max(usage) if usage else 0
            throttled_marker_heights = [u + max_usage * 0.01 for u in throttled_usage]
            ax1.scatter(throttled_times, throttled_marker_heights, marker='o', s=20, 
                       color='darkred', edgecolors='white', linewidth=0.4, zorder=6)
        
        # Labels
        ax1.set_xlabel('Seconds')
        ax1.set_ylabel('Tokens per call')
        ax2.set_ylabel('Tokens in bucket', color='purple')
        ax2.tick_params(axis='y', labelcolor='purple')
        
        info = contract_keys.get(product_id, {})
        ax1.set_title(f'Token Bucket Behavior: {product_id}\n{info.get("description", "")}')
        
        # Stats
        success = sum(code == 200 for code in status_codes)
        throttled = sum(code == 429 for code in status_codes)
        print(f"{product_id}: Calls: {len(status_codes)} | Success: {success} | Throttled: {throttled}")
    
    # Add legend to first subplot
    legend_items = [
        Patch(facecolor='tab:green', alpha=0.7, label='Success (200)'),
        Line2D([0], [0], color='purple', linewidth=2, label='Bucket level'),
        Line2D([0], [0], color='purple', linestyle='--', label='Capacity'),
        Line2D([0], [0], marker='o', color='darkred', markersize=8, linestyle='None',
               markerfacecolor='darkred', markeredgecolor='white', label='Throttled (429)')
    ]
    axes[0, 0].legend(handles=legend_items, loc='upper right', bbox_to_anchor=(0.98, 0.85), framealpha=0.9)
    
    plt.tight_layout()
    plt.show()
else:
    print('Run the load test first to capture api_runs data.')

---
## 📊 Results Summary
---

In [ ]:
# Overall summary
utils.print_info(f"\n{'='*80}")
utils.print_info(f"📊 ACCESS CONTRACTS TEST SUMMARY")
utils.print_info(f"{'='*80}")

utils.print_info(f"\n📜 Access Contracts Tested: {len(contract_keys)}")
for product_id, info in contract_keys.items():
    utils.print_info(f"   • {product_id}: {info['description']}")

if contracts_with_data:
    total_calls = sum(len(runs) for runs in contracts_with_data.values())
    total_success = sum(sum(1 for r in runs if r[2] == 200) for runs in contracts_with_data.values())
    total_throttled = sum(sum(1 for r in runs if r[2] == 429) for runs in contracts_with_data.values())
    total_tokens_all = sum(sum(r[1] for r in runs) for runs in contracts_with_data.values())

    utils.print_info(f"\n📈 Load Test Results:")
    utils.print_info(f"   Total API calls: {total_calls}")
    utils.print_ok(f"   Successful: {total_success}")
    if total_throttled > 0:
        utils.print_info(f"   Throttled: {total_throttled}")
    utils.print_info(f"   Total tokens consumed: {total_tokens_all}")

    if total_success == total_calls:
        utils.print_ok(f"\n✅ All tests passed!")
    else:
        utils.print_info(f"\n📊 Pass rate: {total_success}/{total_calls}")
else:
    utils.print_info("\n⚠️ No load test data available. Run the load test cells first.")

<a id='cleanup'></a>
### 🧹 Cleanup (Optional)

Remove the test access contracts from APIM created during this notebook session.

> **Note:** This will not delete any created secrets in Azure Key Vault or Microsoft Foundry connections.

In [ ]:
# Set to True to delete the access contracts created in this session
cleanup_enabled = False

if cleanup_enabled:
    for result in deployment_results:
        if not result['success']:
            continue
        
        contract = result['contract']
        product_id = f"LLM-{contract['business_unit']}-{contract['use_case_name']}-{contract['environment']}"
        subscription_name = f"{product_id}-SUB-01"
        
        utils.print_info(f"Deleting {product_id}...")
        
        # Delete product and its associated subscriptions
        prod_cmd = f"az apim product delete --resource-group {governance_hub_resource_group} --service-name {apimClientTool.apim_resource_name} --product-id {product_id} --delete-subscriptions true --yes"
        utils.run(prod_cmd, f"Deleted product {product_id}", f"Failed to delete product")
    
    utils.print_ok("Cleanup completed!")
else:
    utils.print_info("Cleanup is disabled. Set cleanup_enabled = True to remove test resources.")